# Different embedding models produce vectors with different dimensions. It’s important to ensure that the embedding size matches the dimension expected by your vector database or index.

Higher-dimensional embeddings can capture more information, but quality depends on the model architecture, training data, and how the vectors are used—not just the size.

# Recommended embedding models:

### General-purpose / strong defaults
- BAAI/bge-small-en-v1.5 → fast, lightweight (~384 dim)
- BAAI/bge-base-en-v1.5 → great balance (~768 dim)
- BAAI/bge-large-en-v1.5 → very strong quality (~1024 dim)

### Multilingual (if you need Hebrew / multiple languages)
- sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

### Retrieval / RAG-focused
- intfloat/e5-small-v2
- intfloat/e5-base-v2
- intfloat/e5-large-v2

In [5]:
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# Make sure to run load_dotenv for setting Huggingface and OpenAIKey token via environment variable (auto process)
load_dotenv()

True

### HuggingFace Embeddings

In [6]:


embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
# Single text embedding:

text = "Embedding this text..."
embedding_text_results = embedding.embed_query(text)
print(f"Embedding length: ${len(embedding_text_results)} - Dimension vector size")
print(embedding_text_results[:10])

Embedding length: $384 - Dimension vector size
[0.018786069005727768, 0.11936674267053604, 0.002703667152673006, 0.013673366978764534, 0.03584182262420654, 0.05444418266415596, 0.09444339573383331, -0.0470210500061512, 0.05289117246866226, -0.04625224694609642]


In [23]:
# Multiply texts embedding:

sentences = ["Embedding this text...", "Embedding this text as well...", "Also this one..."]
embedding_sentences_results = embedding.embed_documents(sentences)
print(f"Embedding length (for sentence): ${len(embedding_sentences_results[0])} - Dimension vector size")
print(embedding_sentences_results[0][:10])

Embedding length (for sentence): $384 - Dimension vector size
[0.01878606155514717, 0.11936674267053604, 0.0027036124374717474, 0.013673385605216026, 0.035841844975948334, 0.054444171488285065, 0.0944434180855751, -0.0470210425555706, 0.052891191095113754, -0.046252280473709106]


### OpenAI Embeddings
https://developers.openai.com/api/docs/guides/embeddings

In [8]:
embedding_model = "text-embedding-3-small"
embedding = OpenAIEmbeddings(model=embedding_model)

In [11]:
# Single text embedding:

single_text = "Langchain is a must framework for building RAG systems"
single_embedding = embedding.embed_query(single_text)
print(f"Embedding length: ${len(single_embedding)} - Dimension vector size")
print(single_embedding[:10])

Embedding length: $1536 - Dimension vector size
[-0.058502197265625, -0.006900787353515625, 0.0260162353515625, -0.0188446044921875, 0.01235198974609375, -0.01392364501953125, -0.002593994140625, -0.01593017578125, -0.00798797607421875, -0.016265869140625]


In [13]:
# Multiply texts embedding:

multiply_texts = [
    "Langchain is a must framework for building RAG systems",
    "Embedding converts text into vectors",
    "Local AI Models are amazing tool for security"
]
multiply_embeddings = embedding.embed_documents(multiply_texts)
print(f"Embedding length: ${len(multiply_embeddings[0])} - Dimension vector size")
print(multiply_embeddings[0][:10])

Embedding length: $1536 - Dimension vector size
[-0.058502197265625, -0.006900787353515625, 0.0260162353515625, -0.0188446044921875, 0.01235198974609375, -0.01392364501953125, -0.002593994140625, -0.01593017578125, -0.00798797607421875, -0.016265869140625]


### Sematic And Similarity Search

In [18]:
# Similarity search

content = [
    "RoadRecorder is a app for tracking after user location in real time",
    "WaveSounds is a digital music player",
    "Python is the best language for AI development",
    "8Invest is an app for trading with crypto currency",
    "Today python is the number one language by devs"
]

def cosine_similarity(vet1, vet2):
    """
        This function computes the cosine similarity between two vectors
        Result close to 1 - Very similar
        Result close to 0 - Not related
        Result close to -1 - Opposite meaning
    """
    dot_item = np.dot(vet1, vet2)
    norm_a = np.linalg.norm(vet1)
    norm_b = np.linalg.norm(vet2)

    return dot_item / (norm_a * norm_b)

In [19]:
embedding_model = "text-embedding-3-small"
embedding = OpenAIEmbeddings(model=embedding_model)

In [20]:
sentence_embeddings = embedding.embed_documents(content)
print(sentence_embeddings[0][:10])

[-0.0340576171875, 0.046234130859375, 0.01947021484375, 0.0111541748046875, -0.088623046875, -0.023529052734375, 0.036163330078125, -0.01373291015625, -0.0029315948486328125, -0.0145111083984375]


In [21]:
for i in range(len(content)):
    for j in range(i + 1, len(content)):
        similarity = cosine_similarity(sentence_embeddings[i], sentence_embeddings[j])

        print(f"{content[i]} vs {content[j]} similarity: {similarity}")

RoadRecorder is a app for tracking after user location in real time vs WaveSounds is a digital music player similarity: 0.2344034373426096
RoadRecorder is a app for tracking after user location in real time vs Python is the best language for AI development similarity: 0.05328434482453224
RoadRecorder is a app for tracking after user location in real time vs 8Invest is an app for trading with crypto currency similarity: 0.19048755037248083
RoadRecorder is a app for tracking after user location in real time vs Today python is the number one language by devs similarity: 0.06867651686604091
WaveSounds is a digital music player vs Python is the best language for AI development similarity: 0.0592290791976913
WaveSounds is a digital music player vs 8Invest is an app for trading with crypto currency similarity: 0.27005484269324154
WaveSounds is a digital music player vs Today python is the number one language by devs similarity: 0.09120533828275633
Python is the best language for AI developmen

In [22]:
# Semantic search

def semantic_search(query, documents, embeddings_models, top_k = 3):
    """
        Simple semantic search implementation
    """

    query_embedding = embeddings_models.embed_query(query)
    doc_embeddings = embeddings_models.embed_documents(documents)

    similarities = []

    for i, doc_emb in enumerate(doc_embeddings):
        similarity_res = cosine_similarity(query_embedding, doc_emb)
        similarities.append((similarity_res, documents[i]))

    similarities.sort(reverse=True)
    return similarities[:top_k]

In [25]:
results = semantic_search("What is RoadRecorder?", content, embedding)
print(results)

[(np.float64(0.6642886051653902), 'RoadRecorder is a app for tracking after user location in real time'), (np.float64(0.24827481671440418), 'WaveSounds is a digital music player'), (np.float64(0.10596928114288195), '8Invest is an app for trading with crypto currency')]
